# 第5周 · 课堂练习

把 `raise NotImplementedError` 换成你的代码，再运行检查。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from zero2researcher import data
from zero2researcher.checks import week05 as ck

## E1 · 混淆矩阵手算

下面是 10 个样本的真实标签和模型预测（1 = 阳性）。**先在纸上**数出 TP、FP、FN、TN，再把四个数填进代码。

In [ ]:
y_true = [1, 1, 1, 1, 0, 0, 0, 0, 0, 0]
y_pred = [1, 1, 0, 1, 0, 0, 1, 0, 0, 0]

# 在这里写你的代码
raise NotImplementedError
ck.confusion_values(tp, fp, fn, tn)
print("precision =", tp / (tp + fp), "   recall =", tp / (tp + fn))

## E2 · 指标选择判断题（markdown 作答）

下面三个场景各应优先看哪个指标？为什么？（每题一两句话）

1. 罕见病初筛：漏掉一个真病人，可能错过治疗窗口；筛出假阳性只需复检一次。
2. 垃圾邮件过滤：把老板的邮件错标成垃圾（删掉），比放过一封垃圾邮件严重得多。
3. 95:5 的不平衡数据上比较两个分类器的总体好坏。

（在这里回答）

## E3 · 分层交叉验证补全

照课上的写法：Titanic 的 `pclass / age / fare / sex` 四个特征，`SimpleImputer(中位数) → StandardScaler → LogisticRegression(max_iter=1000)` 的 Pipeline，`StratifiedKFold(5, shuffle=True, random_state=42)`，算 5 折准确率。

自查时把 Pipeline 和交给 `cross_val_score` 的 `X` 一起传进去：`ck.titanic_cv_scores(scores, pipe, X)`——只看分数分辨不出预处理是否放在了交叉验证外面。

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

titanic = data.load_titanic()
X = titanic[["pclass", "age", "fare"]].copy()
X["sex"] = (titanic["sex"] == "female").astype(int)
y = titanic["survived"]

# 在这里写你的代码
raise NotImplementedError
print(np.round(scores, 3), f"平均 {scores.mean():.3f}")
ck.titanic_cv_scores(scores, pipe, X)

## E4 · 把 ESL 的"错误交叉验证"改对

下面给出课上重演的**错误流程**（完整可运行）。你的任务：写出**正确流程**——特征选择放进每一折内部，报告 5 折平均误差 `right_err`。

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

rng = np.random.default_rng(42)
X_rand = rng.standard_normal((50, 5000))
y_rand = rng.permutation(np.array([0] * 25 + [1] * 25))


def top_corr_features(Xs, ys, k=100):
    """返回与标签相关系数绝对值最大的 k 个特征的列号"""
    Xc = Xs - Xs.mean(axis=0)
    yc = ys - ys.mean()
    corr = (Xc * yc[:, None]).sum(axis=0) / (
        np.sqrt((Xc ** 2).sum(axis=0)) * np.sqrt((yc ** 2).sum()) + 1e-12)
    return np.argsort(-np.abs(corr))[:k]


# ---- 错误流程（特征和标签毫无关系，误差却低得离谱）----
sel = top_corr_features(X_rand, y_rand)
knn = KNeighborsClassifier(n_neighbors=1)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
wrong_err = 1 - cross_val_score(knn, X_rand[:, sel], y_rand, cv=skf).mean()
print(f"错误流程的 CV 误差：{wrong_err:.1%}")

In [ ]:
# 在这里写你的代码
raise NotImplementedError
print(f"正确流程的 CV 误差：{right_err:.1%}")
ck.right_cv_error(right_err)

## E5 · 技术重复不是生物学重复：按组划分

下面模拟一个常见的实验设计：30 只小鼠，每只测 8 个技术重复孔，每孔 4 个读数。标签（处理组 / 对照组）按小鼠分配，而且**和读数毫无关系**——任何模型的真实准确率都只能约 0.5。但每只小鼠的读数有自己的"指纹"，同一只小鼠的 8 个孔彼此很像。

用给出的 `knn5`（`KNeighborsClassifier(n_neighbors=5)`）分别做两种 5 折交叉验证，报告平均准确率：

1. `acc_record`：`StratifiedKFold(5, shuffle=True, random_state=0)`——按孔随机拆；
2. `acc_group`：`GroupKFold(5, shuffle=True, random_state=0)`，并把 `groups=mouse_id` 传给 `cross_val_score`——同一只小鼠的孔永远落在同一侧。

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.neighbors import KNeighborsClassifier

rng = np.random.default_rng(2027)
mouse_label = rng.permutation(np.array([0, 1] * 15))       # 30 只小鼠：处理组 / 对照组各 15 只
mouse_profile = rng.normal(0, 1, size=(30, 4))             # 每只小鼠自己的"指纹"，与组别无关
mouse_id = np.repeat(np.arange(30), 8)                     # 每只小鼠 8 个技术重复孔
X_mice = mouse_profile[mouse_id] + rng.normal(0, 0.3, size=(240, 4))
y_mice = mouse_label[mouse_id]
knn5 = KNeighborsClassifier(n_neighbors=5)

# 在这里写你的代码
raise NotImplementedError
print(f"按孔拆：{acc_record:.3f}   按小鼠拆：{acc_group:.3f}")
ck.group_split_accs(acc_record, acc_group)

**问题**：两个数字为什么差这么多？如果要把这个模型用在**一批新小鼠**上，该相信哪一个？你自己的实验数据里，有哪些这样的"组"？（三四句话）

（在这里回答）